# BronchoMAS Unified Test Notebook

This notebook combines the **three current pipelines** in one place:

1. **Runtime** — fast real-time coaching
2. **MAS / Research** — multi-agent line
3. **SAS** — single-agent-with-skills line

It is adapted to the **current repo layout**:

- `broncho_mas.runtime.manager.RuntimeManager`
- `broncho_mas.mas.manager.MASManager`
- `broncho_mas.sas.manager.SASManager`
- adapter entry: `broncho_mas.SmolAgentsLLM`

It also includes an **OpenAI configuration section** so you can later switch from HF to OpenAI without rewriting the notebook.


In [1]:
# --- Common imports ---
import os
import json
import getpass
from copy import deepcopy
from pathlib import Path
from pprint import pprint

# --- Setup project path ---
import sys
from pathlib import Path

project_root = Path.cwd()
src_path = project_root / "src"

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print("project_root =", project_root)
print("src_path     =", src_path)
print("src exists   =", src_path.exists())


project_root = C:\Users\LAB-ADMIN\Documents\GitHub\broncho_MAS\bronchoscopy_AI_assistant
src_path     = C:\Users\LAB-ADMIN\Documents\GitHub\broncho_MAS\bronchoscopy_AI_assistant\src
src exists   = True


## 0) Backend setup

By default this notebook keeps **HF** as the active backend. When testing other models, please change this block accordingly. 

In [2]:
# --- Default backend setup ---
os.environ["BRONCHO_PROVIDER"] = "openai"
os.environ["BRONCHO_MODEL"] = "gpt-5.4-nano-2026-03-17"

provider = os.environ.get("BRONCHO_PROVIDER", "").strip().lower()

if provider == "hf":
    if not os.environ.get("HF_TOKEN"):
        try:
            token = getpass.getpass("Enter HF_TOKEN (leave blank to skip): ").strip()
            if token:
                os.environ["HF_TOKEN"] = token
        except Exception:
            pass

elif provider == "openai":
    if not os.environ.get("OPENAI_API_KEY"):
        try:
            token = getpass.getpass("Enter OPENAI_API_KEY (leave blank to skip): ").strip()
            if token:
                os.environ["OPENAI_API_KEY"] = token
        except Exception:
            pass

print("BRONCHO_PROVIDER    =", os.environ.get("BRONCHO_PROVIDER"))
print("BRONCHO_MODEL       =", os.environ.get("BRONCHO_MODEL"))
print("BRONCHO_TOOL_CHOICE =", os.environ.get("BRONCHO_TOOL_CHOICE"))
print("HF token set        =", bool(os.environ.get("HF_TOKEN")))
print("OpenAI key set      =", bool(os.environ.get("OPENAI_API_KEY")))


Enter OPENAI_API_KEY (leave blank to skip):  ········


BRONCHO_PROVIDER    = openai
BRONCHO_MODEL       = gpt-5.4-nano-2026-03-17
BRONCHO_TOOL_CHOICE = None
HF token set        = False
OpenAI key set      = True


In [3]:
# --- Imports from current codebase ---
print("Testing imports...")

import broncho_mas
from broncho_mas import SmolAgentsLLM, RuntimeManager, SASManager

mas_available = True
mas_import_error = None
MASManager = None

try:
    from broncho_mas import MASManager
except Exception as e:
    mas_available = False
    mas_import_error = repr(e)

print("broncho_mas imported from:", broncho_mas.__file__)
print("RuntimeManager:", RuntimeManager)
print("SASManager    :", SASManager)
print("MAS available :", mas_available)
if not mas_available:
    print("MAS import error:", mas_import_error)


Testing imports...
broncho_mas imported from: C:\Users\LAB-ADMIN\Documents\GitHub\broncho_MAS\bronchoscopy_AI_assistant\src\broncho_mas\__init__.py
RuntimeManager: <class 'broncho_mas.runtime.manager.RuntimeManager'>
SASManager    : <class 'broncho_mas.sas.manager.SASManager'>
MAS available : True


## 1) Shared payload helper

In [4]:
def make_payload(**overrides):
    payload = {
        "iteration": 1,
        "mode": "Automatic",
        "visualization_mode": "Raw",
        "robot_joints": [0.0, 0.0, 0.0],
        "m_jointsVelRel": [0.0, 0.0, 0.0],
        "bounding_boxes": [],
        "current_airway": "RB2",
        "target_airway": "RB3",
        "requested_next_airway": "RB3",
        "reached_regions": ["RB1", "RB2"],
        "phase": "navigation",
        "need_llm": True,
        "llm_reason": "Need short live coaching for next navigation step.",
        "soft_prompt": "Short coaching sentence for bronchoscopy navigation.",
        "previous_msgs": "Hold center. Keep the lumen visible.",
        "student_question": "Can we talk about the weather",
        "is_centered": True,
        "is_stable": True,
        "is_target_visible": False,
        "drift_detected": False,
        "wall_contact_risk": False,
        "need_recenter": False,
        "meta": {
            "source": "notebook_test",
            "note": "synthetic payload for manager tests",
        },
    }
    payload.update(overrides)
    return payload

def print_result_compact(title: str, result: dict):
    raw = result.get("raw", {}) if isinstance(result, dict) else {}
    meta = result.get("statepacket", {}).get("meta", {}) if isinstance(result, dict) else {}
    normalized = raw.get("normalized_state", {}) if isinstance(raw, dict) else {}

    print(f"--- {title} ---")
    print("  manager              :", meta.get("manager"))
    print("  model                :", meta.get("model"))
    print("  ui_text              :", result.get("ui_text"))
    print("  llm_ui               :", raw.get("llm_ui"))
    print("  deterministic_ui     :", raw.get("deterministic_ui"))
    print("  m_jointsVelRel       :", normalized.get("m_jointsVelRel"))
    print("  need_llm             :", normalized.get("need_llm"))
    print("  llm_reason           :", normalized.get("llm_reason"))
    print("  used_fallback_backend:", raw.get("used_fallback_backend"))
    print()

sample_payload = make_payload()
pprint(sample_payload)


{'bounding_boxes': [],
 'current_airway': 'RB2',
 'drift_detected': False,
 'is_centered': True,
 'is_stable': True,
 'is_target_visible': False,
 'iteration': 1,
 'llm_reason': 'Need short live coaching for next navigation step.',
 'm_jointsVelRel': [0.0, 0.0, 0.0],
 'meta': {'note': 'synthetic payload for manager tests',
          'source': 'notebook_test'},
 'mode': 'Automatic',
 'need_llm': True,
 'need_recenter': False,
 'phase': 'navigation',
 'previous_msgs': 'Hold center. Keep the lumen visible.',
 'reached_regions': ['RB1', 'RB2'],
 'requested_next_airway': 'RB3',
 'robot_joints': [0.0, 0.0, 0.0],
 'soft_prompt': 'Short coaching sentence for bronchoscopy navigation.',
 'student_question': 'Can we talk about the weather',
 'target_airway': 'RB3',
 'visualization_mode': 'Raw',
 'wall_contact_risk': False}


## 2) Instantiate the three lines


In [5]:
runtime = RuntimeManager()
sas = SASManager()

mas = None
if mas_available and MASManager is not None:
    try:
        mas = MASManager()
    except Exception as e:
        print("MASManager init failed:", repr(e))

print("runtime =", type(runtime).__name__)
print("sas     =", type(sas).__name__)
print("mas     =", type(mas).__name__ if mas is not None else None)


[runtime] provider=openai model=gpt-5.4-nano-2026-03-17
[broncho_mas] log dir = C:\Users\LAB-ADMIN\Documents\GitHub\broncho_MAS\bronchoscopy_AI_assistant\mas_log\20260323_074532_682780
[sas] provider=openai model=gpt-5.4-nano-2026-03-17
[broncho_mas] log dir = C:\Users\LAB-ADMIN\Documents\GitHub\broncho_MAS\bronchoscopy_AI_assistant\mas_log\20260323_074533_104496
[broncho_mas] log dir = C:\Users\LAB-ADMIN\Documents\GitHub\broncho_MAS\bronchoscopy_AI_assistant\mas_log\20260323_074533_572950
[broncho_mas manager] version=0.0.5-mas mas_mode=agentic provider=openai model=gpt-5.4-nano-2026-03-17
runtime = RuntimeManager
sas     = SASManager
mas     = MASManager


## 3) Runtime tests


In [ ]:
runtime_cases = [
    ("basic_next_step", make_payload()),
    (
        "recenter_case",
        make_payload(
            iteration=2,
            current_airway="RB3",
            target_airway="RB4",
            requested_next_airway="RB4",
            reached_regions=["RB1", "RB2", "RB3"],
            m_jointsVelRel=[-0.2, 0.1, 0.0],
            drift_detected=True,
            need_recenter=True,
            is_centered=False,
            previous_msgs="Advance carefully.",
            student_question="I think I am drifting. What should I do?",
        ),
    ),
    (
        "target_visible_case",
        make_payload(
            iteration=3,
            current_airway="RB4",
            target_airway="RB5",
            requested_next_airway="RB5",
            reached_regions=["RB1", "RB2", "RB3", "RB4"],
            is_target_visible=True,
            m_jointsVelRel=[0.1, -0.1, 0.0],
            student_question="I can see the next opening now.",
        ),
    ),
]

runtime_results = []
for name, payload in runtime_cases:
    print("\n" + "=" * 80)
    print("RUNTIME CASE:", name)
    result = runtime.step(deepcopy(payload))
    runtime_results.append((name, result))
    print_result_compact("runtime summary", result)


## 4) MAS / Research tests

This uses the **current** `MASManager.step(payload)` path, not the old `research/manager.py` path.


In [ ]:
mas_results = []

if mas is None:
    print("MAS line not available in this environment.")
else:
    mas_cases = [
        ("mas_basic", make_payload()),
        (
            "mas_followup",
            make_payload(
                iteration=2,
                current_airway="LB1+2",
                target_airway="LB3",
                requested_next_airway="LB3",
                reached_regions=["RB1", "RB2", "LB1+2"],
                previous_msgs="You already reached the left upper entry.",
                student_question="How should I proceed toward LB3?",
                m_jointsVelRel=[0.0, 0.2, 0.0],
            ),
        ),
    ]

    for name, payload in mas_cases:
        print("\n" + "=" * 80)
        print("MAS CASE:", name)
        result = mas.step(deepcopy(payload))
        mas_results.append((name, result))
        print_result_compact("mas summary", result)


## 5) SAS tests

This keeps both:
- direct manager path: `SASManager.step(payload)`
- prompt-only compatibility path


In [7]:
sas_cases = [
    ("sas_basic", make_payload()),
    (
        "sas_safety_case",
        make_payload(
            iteration=2,
            current_airway="RB6",
            target_airway="RB7",
            requested_next_airway="RB7",
            reached_regions=["RB1", "RB2", "RB3", "RB4", "RB5", "RB6"],
            previous_msgs="You are close to the bifurcation.",
            student_question="Can WE TALK ABOUT THE WEATHER",
            wall_contact_risk=True,
            need_recenter=True,
            drift_detected=True,
            is_centered=False,
            is_stable=False,
            m_jointsVelRel=[-0.3, 0.0, 0.0],
        ),
    ),
]

print("Defined SAS cases:", [name for name, _ in sas_cases])


Defined SAS cases: ['sas_basic', 'sas_safety_case']


In [9]:
from copy import deepcopy

sas_results = []
for name, payload in sas_cases:
    print("\n" + "=" * 80)
    print("SAS CASE:", name)

    result = sas.step(deepcopy(payload))
    sas_results.append((name, result))

    raw_payload = result.get("raw", {}).get("normalized_state", {}).get("raw_payload", {})
    state = result.get("raw", {}).get("normalized_state", {})

    print_result_compact("sas summary", result)
    print("  raw_payload.m_jointsVelRel:", raw_payload.get("m_jointsVelRel"))
    print("  state.m_jointsVelRel      :", state.get("m_jointsVelRel"))
    print()



SAS CASE: sas_basic
--- sas summary ---
  manager              : SASManager
  model                : gpt-5.4-nano-2026-03-17
  ui_text              : Keep neutral; re-anchor at the carina, then advance into RB3 in small increments.
  llm_ui               : Keep neutral; re-anchor at the carina, then advance into RB3 in small increments.
  deterministic_ui     : Nice. Stay centered and find the anterior segment of the right upper lobe by carina bifurcation; symmetric main bronchi.
  m_jointsVelRel       : [0.0, 0.0, 0.0]
  need_llm             : True
  llm_reason           : Need short live coaching for next navigation step.
  used_fallback_backend: False

  raw_payload.m_jointsVelRel: [0.0, 0.0, 0.0]
  state.m_jointsVelRel      : [0.0, 0.0, 0.0]


SAS CASE: sas_safety_case
--- sas summary ---
  manager              : SASManager
  model                : gpt-5.4-nano-2026-03-17
  ui_text              : Tilt down the knob, then re-anchor at the carina; reset to neutral and center RB7.
  

## 6) Legacy prompt compatibility check

In [ ]:
legacy_prompt = '''
CURRENT_SITUATION: Current airway: RB2
Target airway: RB3
reached_regions: ["RB1", "RB2"]
m_jointsVelRel: [0.1, 0.0, 0.0]
Need LLM: true

PREVIOUS_MSGS: Hold steady. Keep the lumen centered.

STUDENT_QUESTION: Where do I go next?
'''.strip()

print("Legacy prompt compatibility check:")
legacy_result = sas.run(legacy_prompt)
print_result_compact("legacy sas summary", legacy_result)
